# Regressione con scikit-learn e statsmodels: quattro algoritmi, quattro dataset

In questo notebook applichiamo quattro algoritmi predittivi di regressione ad altrettanti dataset
sintetici, ispirati ai dati che un'azienda software produce ogni giorno (issue tracker, pipeline CI,
bolletta cloud, sprint). Ogni algoritmo viene applicato a **un solo** dataset, scelto in modo che le
caratteristiche dei dati mettano in luce i punti di forza (e i limiti) del metodo.

| Sezione | Dataset | Target | Algoritmo | Perché proprio qui |
|---|---|---|---|---|
| 3 | `costo_cloud.csv` | `costo_euro` | **Regressione lineare multivariata** (scikit-learn + **statsmodels** per l'inferenza) | il costo è, per costruzione, una somma di voci a prezzo unitario: il modello lineare è *il* modello giusto e i coefficienti si leggono come un listino |
| 4 | `ticket_risoluzione.csv` | `ore_risoluzione` | **Albero decisionale** | molte variabili categoriche (priorità, componente, tipo): l'albero le gestisce senza scaling e produce regole leggibili come un `if/else` |
| 5 | `story_point_ore.csv` | `ore_effettive` | **k-Nearest Neighbors** | "quanto ci hanno messo le story simili a questa?" è esattamente la domanda che il kNN risponde |
| 6 | `build_ci.csv` | `durata_minuti` | **Rete neurale MLP** | interazioni (cache × dimensione del commit) e effetti non lineari (ore di punta): la rete li impara da sola |

**Come misuriamo la qualità.** Per ogni modello riportiamo **MSE** (errore quadratico medio) e **RMSE**
(la sua radice, nelle unità del target) calcolati su dati **mai visti in addestramento**. Usiamo due
strumenti complementari:

- un **test set separato** (`train_test_split`, oppure una separazione temporale per la serie storica dei costi cloud), che tocchiamo una sola volta, alla fine;
- la **cross-validazione a 5 fold** sul solo training set, per scegliere gli iperparametri senza "consumare" il test.

Per dare un senso ai numeri, confrontiamo sempre l'RMSE del modello con quello di un **modello ingenuo**
che predice la media del training: se non facciamo meglio di così, il modello non serve.

**Struttura.** Prima analizziamo ogni dataset (dimensioni, tipi, valori mancanti, distribuzione del
target, correlazioni, relazioni con le variabili categoriche); poi definiamo le scelte comuni di
preparazione; infine costruiamo, ottimizziamo e valutiamo i quattro modelli, chiudendo con un riepilogo.

**Requisiti ed esecuzione in VS Code.** Servono `pandas`, `numpy`, `matplotlib`, `seaborn`,
`scikit-learn` (≥ 1.2) e `statsmodels`. In un ambiente conda:

```
conda install pandas numpy matplotlib seaborn scikit-learn statsmodels ipykernel
```

Apriamo il notebook in VS Code, scegliamo il kernel dell'ambiente (pulsante *Select Kernel* in alto
a destra) e usiamo *Run All*. I quattro file CSV devono trovarsi nella stessa cartella del notebook
(oppure modifichiamo `DATA_DIR` nella cella di setup).

## 0. Setup: librerie, configurazione e funzioni di supporto

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, GridSearchCV, cross_val_score
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, FunctionTransformer
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor, plot_tree, export_text
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, make_scorer

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 4.5)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

RANDOM_STATE = 42              # seme unico: risultati riproducibili in ogni cella
DATA_DIR = Path(".")           # cartella che contiene i quattro CSV

ModuleNotFoundError: No module named 'seaborn'

`statsmodels` lo importiamo a parte: lo usiamo soltanto nella sezione 3, per l'**inferenza** sulla
regressione lineare (errori standard, intervalli di confidenza, p-value, diagnostica dei residui),
cioè per tutto ciò che scikit-learn, orientato alla sola previsione, non fornisce.

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor

Definiamo tre funzioni che riutilizzeremo per tutti i dataset e tutti i modelli, così da non
ripetere codice e da essere certi di calcolare le metriche sempre nello stesso modo.

In [ ]:
def metriche(y_vero, y_pred, etichetta=""):
    '''Calcola MSE e RMSE tra valori reali e previsti; restituisce un dizionario
    comodo da accumulare in una tabella di riepilogo.'''
    mse = mean_squared_error(y_vero, y_pred)
    return {"modello": etichetta, "MSE": mse, "RMSE": np.sqrt(mse)}


def rmse_baseline(y_train, y_test):
    '''RMSE del modello ingenuo che predice sempre la media del training:
    è la pietra di paragone minima per qualunque modello.'''
    return np.sqrt(mean_squared_error(y_test, np.full(len(y_test), y_train.mean())))


def riepilogo(df, nome):
    '''Stampa dimensioni, tipi e valori mancanti; restituisce le statistiche descrittive.'''
    print(f"=== {nome}: {df.shape[0]} righe x {df.shape[1]} colonne ===\n")
    print("Tipi di dato:")
    print(df.dtypes.to_string(), "\n")
    print("Valori mancanti per colonna:")
    print(df.isna().sum().to_string(), "\n")
    print("Statistiche descrittive (numeriche e categoriche):")
    return df.describe(include="all").T


def distribuzione_target(y, nome, unita):
    '''Istogramma del target in scala originale e logaritmica: ci dice se conviene
    modellare log(y) invece di y.'''
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
    axes[0].hist(y, bins=40, color="steelblue")
    axes[0].set_title(f"{nome}: scala originale")
    axes[0].set_xlabel(unita)
    axes[1].hist(np.log(y), bins=40, color="darkorange")
    axes[1].set_title(f"log({nome})")
    axes[1].set_xlabel(f"log({unita})")
    plt.tight_layout()
    plt.show()
    print(f"media = {y.mean():.2f}   mediana = {y.median():.2f}   "
          f"minimo = {y.min():.2f}   massimo = {y.max():.2f}   asimmetria = {y.skew():.2f}")


def heatmap_correlazioni(df, titolo):
    '''Matrice di correlazione di Pearson tra le sole colonne numeriche.'''
    corr = df.select_dtypes("number").corr()
    fig, ax = plt.subplots(figsize=(1.1 * len(corr) + 2, 0.8 * len(corr) + 1.5))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, ax=ax)
    ax.set_title(titolo)
    plt.tight_layout()
    plt.show()

Carichiamo i quattro dataset. L'unica trasformazione che facciamo subito è convertire la colonna
`mese` dei costi cloud in una data vera e propria, così da poter disegnare serie storiche.

In [ ]:
cloud = pd.read_csv(DATA_DIR / "costo_cloud.csv")
ticket = pd.read_csv(DATA_DIR / "ticket_risoluzione.csv")
story = pd.read_csv(DATA_DIR / "story_point_ore.csv")
build = pd.read_csv(DATA_DIR / "build_ci.csv")

cloud["mese"] = pd.to_datetime(cloud["mese"], format="%Y-%m")

for nome, df in {"costo_cloud": cloud, "ticket_risoluzione": ticket,
                 "story_point_ore": story, "build_ci": build}.items():
    print(f"{nome:20s} {df.shape[0]:5d} righe x {df.shape[1]:2d} colonne")

## 1. Analisi esplorativa dei dati

Prima di costruire qualunque modello guardiamo i dati. Per ogni dataset rispondiamo alle stesse
domande:

1. **Quanti dati abbiamo e di che tipo?** Dimensioni, tipi di colonna, valori mancanti.
2. **Com'è fatto il target?** Se è fortemente asimmetrico (poche osservazioni enormi), un modello che
   minimizza l'errore quadratico insegue quelle poche righe: in quel caso lavoriamo su `log(y)`.
3. **Quali variabili sono legate al target, e tra loro?** Correlazioni di Pearson tra le numeriche
   (attenzione: misurano solo relazioni *lineari*) e boxplot del target per le categoriche.
4. **Cosa ci suggerisce tutto questo sul modello?** Trasformazioni, codifiche, feature da escludere.

Le decisioni di preparazione dei dati che prenderemo nella sezione 2 nascono da qui.

### 1.1 `costo_cloud`: costo mensile per servizio

Un pannello: 6 servizi osservati per 48 mesi (settembre 2022 – agosto 2026). Le feature sono le
grandezze che "consumano" risorse: utenti attivi, richieste API, dati archiviati, deploy.

In [ ]:
riepilogo(cloud, "costo_cloud")

In [ ]:
distribuzione_target(cloud["costo_euro"], "costo_euro", "euro/mese")

# Andamento nel tempo: trend di crescita e stagionalità, servizio per servizio e in totale
serie = cloud.pivot(index="mese", columns="servizio", values="costo_euro")
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
serie.plot(ax=axes[0])
axes[0].set_title("Costo mensile per servizio")
axes[0].set_ylabel("euro")
serie.sum(axis=1).plot(ax=axes[1], color="black")
axes[1].set_title("Costo totale mensile (tutti i servizi)")
axes[1].set_ylabel("euro")
plt.tight_layout()
plt.show()

In [ ]:
# Stagionalità degli utenti (media per mese dell'anno) e relazione costo ~ richieste
cloud["mese_anno"] = cloud["mese"].dt.month
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
cloud.groupby("mese_anno")["utenti_attivi"].mean().plot.bar(ax=axes[0], color="steelblue")
axes[0].set_title("Utenti attivi medi per mese dell'anno: la stagionalità")
axes[0].set_xlabel("mese dell'anno")
sns.scatterplot(data=cloud, x="richieste_api_milioni", y="costo_euro", hue="servizio", ax=axes[1])
axes[1].set_title("Costo vs richieste API: rette diverse per servizio")
plt.tight_layout()
plt.show()

heatmap_correlazioni(cloud.drop(columns=["mese"]), "Correlazioni: costo_cloud")

**Cosa osserviamo.**

- Il target è moderatamente asimmetrico ma non ha code estreme: possiamo modellarlo in scala
  originale, così i coefficienti restano in **euro**.
- Ogni servizio ha un costo base diverso e una pendenza simile rispetto alle richieste: è la firma
  di un modello **lineare con intercette diverse per servizio** (variabili dummy).
- `utenti_attivi` e `richieste_api_milioni` sono correlate quasi perfettamente (le richieste derivano
  dagli utenti): inserirle entrambe crea **collinearità**, che vedremo con statsmodels.
- La stagionalità (picco autunnale, calo estivo) passa attraverso utenti e richieste: non serve una
  variabile "mese dell'anno" se usiamo già le richieste.
- Nella serie totale spuntano alcuni mesi anomali (picchi isolati): li ritroveremo nei residui.

### 1.2 `ticket_risoluzione`: tempo di risoluzione di un ticket

1.500 ticket con priorità, componente, tipo, dimensione della descrizione e informazioni
sull'assegnatario. Il target sono le ore tra apertura e chiusura.

In [ ]:
riepilogo(ticket, "ticket_risoluzione")

In [ ]:
distribuzione_target(ticket["ore_risoluzione"], "ore_risoluzione", "ore")

ordine_prio = ["Bassa", "Media", "Alta", "Critica"]
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
sns.boxplot(data=ticket, x="priorita", y="ore_risoluzione", order=ordine_prio, ax=axes[0])
sns.boxplot(data=ticket, x="tipo", y="ore_risoluzione", ax=axes[1])
sns.boxplot(data=ticket, x="componente", y="ore_risoluzione", ax=axes[2])
for ax in axes:
    ax.set_yscale("log")          # scala log: altrimenti i pochi ticket "dimenticati" schiacciano tutto
axes[2].tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

print("Mediana delle ore per priorità:")
print(ticket.groupby("priorita")["ore_risoluzione"].median().reindex(ordine_prio).round(1).to_string())

In [ ]:
heatmap_correlazioni(ticket, "Correlazioni (numeriche): ticket_risoluzione")

**Cosa osserviamo.**

- Il target è **molto asimmetrico**: mediana intorno alle 40 ore, ma una coda di ticket da centinaia
  o migliaia di ore (i ticket "dimenticati"). In scala logaritmica la distribuzione diventa quasi
  simmetrica: modelleremo `log(ore)` e riporteremo le metriche in ore.
- Le variabili **categoriche** contano più delle numeriche: la priorità ordina nettamente i tempi
  (Critica < Alta < Media < Bassa), le richieste di funzionalità durano più dei bug e le domande
  meno, Infrastruttura e Database sono i componenti più lenti.
- Le correlazioni lineari delle numeriche con il target sono deboli: non perché le variabili siano
  inutili, ma perché gli effetti sono moltiplicativi e mascherati dagli outlier. Un modello che
  lavora per **soglie e combinazioni**, come l'albero, è una scelta naturale.

### 1.3 `story_point_ore`: dalle stime in punti alle ore effettive

800 user story di quattro team: per ognuna conosciamo la stima in story point, il tipo, la seniority
di chi l'ha sviluppata, il numero di criteri di accettazione e la presenza di dipendenze esterne.

In [ ]:
riepilogo(story, "story_point_ore")

In [ ]:
distribuzione_target(story["ore_effettive"], "ore_effettive", "ore")

per_punto = story.groupby("story_point")["ore_effettive"].agg(media="mean", mediana="median",
                                                               dev_std="std", n="size")
per_punto["ore_per_singolo_punto"] = per_punto["media"] / per_punto.index

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
sns.boxplot(data=story, x="story_point", y="ore_effettive", ax=axes[0])
axes[0].set_yscale("log")
axes[0].set_title("Ore effettive per story point (scala log)")
per_punto["ore_per_singolo_punto"].plot.bar(ax=axes[1], color="darkorange")
axes[1].set_title("Ore medie per singolo punto: crescono con la dimensione della story")
axes[1].set_ylabel("ore / punto")
plt.tight_layout()
plt.show()

per_punto.round(1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
sns.boxplot(data=story, x="team", y="ore_effettive", ax=axes[0])
axes[0].set_yscale("log")
axes[0].set_title("Per team: a parità di punti, tempi diversi")
sns.boxplot(data=story, x="seniority_sviluppatore", y="ore_effettive",
            order=["Junior", "Mid", "Senior"], ax=axes[1])
axes[1].set_yscale("log")
axes[1].set_title("Per seniority di chi sviluppa")
plt.tight_layout()
plt.show()

heatmap_correlazioni(story, "Correlazioni (numeriche): story_point_ore")

**Cosa osserviamo.**

- La relazione punti → ore **non è proporzionale**: le ore per singolo punto crescono con la
  dimensione della story. È l'errore sistematico delle stime umane: le story grandi vengono
  sottostimate. Un modello lineare "ore = k · punti" sbaglierebbe in modo prevedibile.
- La **variabilità cresce con i punti** (eteroschedasticità): una story da 13 punti può durare
  da 40 a 400 ore. In scala log la dispersione è più omogenea.
- Il **team** conta: gli stessi punti "pesano" quasi il doppio in un team rispetto a un altro.
  Anche seniority e dipendenze esterne spostano i tempi.
- `sprint` non è correlato con nulla: è un identificativo temporale, non una causa. Lo escludiamo,
  anche perché il kNN soffre le variabili irrilevanti (allontanano vicini che in realtà sono simili).

### 1.4 `build_ci`: durata di una pipeline di integrazione continua

2.000 esecuzioni della pipeline: dimensione del commit (righe, file, moduli), test eseguiti,
riutilizzo della cache, ora e giorno di avvio, tipo di runner, evento che ha lanciato la build.

In [ ]:
riepilogo(build, "build_ci")

In [ ]:
distribuzione_target(build["durata_minuti"], "durata_minuti", "minuti")

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
sns.scatterplot(data=build, x="numero_test", y="durata_minuti", hue="cache_hit",
                alpha=0.5, s=15, ax=axes[0])
axes[0].set_title("Durata vs numero di test (colore = cache riutilizzata)")
build.groupby("ora_giorno")["durata_minuti"].mean().plot.bar(ax=axes[1], color="steelblue")
axes[1].set_title("Durata media per ora di avvio: le ore di punta")
axes[1].set_ylabel("minuti")
sns.boxplot(data=build, x="runner", y="durata_minuti", order=["small", "medium", "large"], ax=axes[2])
axes[2].set_title("Per tipo di runner")
plt.tight_layout()
plt.show()

heatmap_correlazioni(build.drop(columns=["id_build"]), "Correlazioni (numeriche): build_ci")

**Cosa osserviamo.**

- Il target è asimmetrico ma senza outlier estremi (da 3 a ~50 minuti): possiamo lasciarlo in
  minuti. Gli effetti (runner, congestione, cache) sono però **moltiplicativi**, cioè non lineari in
  scala originale: un modello lineare in minuti fatica, una rete neurale può impararli da sola.
- `numero_test` è la variabile più correlata con la durata; `righe_modificate`, `file_modificati` e
  `moduli_toccati` sono fortemente correlate **tra loro** (un commit grande lo è in tutte le
  dimensioni): per un modello lineare è un problema di interpretazione, per una rete neurale no.
- Ci sono **effetti non lineari e interazioni**: la cache incide più sui commit grandi; la durata
  cresce nelle ore di punta (tarda mattinata e metà pomeriggio) ma non in modo monotono con l'ora.
  Sono proprio le strutture che un modello lineare fatica a rappresentare e che una rete neurale
  impara da sola.
- `righe_modificate` e `file_modificati` hanno code lunghissime: prima di darle a una rete neurale
  le comprimiamo con `log1p` e poi le standardizziamo.

## 2. Preparazione comune: split, pipeline, codifiche, scala del target

Alcune scelte valgono per tutti i modelli; le fissiamo una volta sola.

**Separare prima, trasformare dopo.** Ogni trasformazione che "impara" dai dati (media e deviazione
standard per lo scaling, categorie per il one-hot) deve essere stimata sul solo training. Se la
calcolassimo su tutto il dataset, informazioni del test filtrerebbero nell'addestramento (*data
leakage*) e le metriche sarebbero ottimistiche. Per garantirlo usiamo sempre una `Pipeline` di
scikit-learn: `fit` sul training stima le trasformazioni e addestra il modello, `predict` sul test le
riapplica senza ristimarle.

**Codifica delle variabili categoriche.**
- `OneHotEncoder`: una colonna 0/1 per ogni categoria (componente, team, runner...).
- `OrdinalEncoder` con ordine esplicito: per la priorità dei ticket, che ha un ordine naturale
  (Bassa < Media < Alta < Critica) che vogliamo conservare.

**Scaling delle numeriche.** Indispensabile per kNN (che misura distanze) e per la rete neurale (che
ottimizza con il gradiente); inutile per l'albero, che confronta ogni variabile solo con se stessa.

**Target in scala logaritmica.** Dove il target ha code lunghe e outlier (ticket, story) addestriamo
il modello su `log(y)` ma vogliamo previsioni e metriche in ore. `TransformedTargetRegressor` fa
esattamente questo: applica `log` prima di `fit` ed `exp` dopo `predict`, così MSE e RMSE risultano
sempre nelle unità originali. Per i costi cloud e le durate delle build, che non hanno outlier
estremi, restiamo nella scala originale.

**Metriche e validazione.** MSE e RMSE sul test set, sempre nelle unità del target. Per la scelta
degli iperparametri usiamo l'RMSE medio in cross-validazione a 5 fold (`KFold` con rimescolamento e
seme fisso, lo stesso per tutti i modelli). Sui due dataset con outlier estremi (ticket e story) la
selezione la facciamo con l'**RMSE in scala logaritmica**: in ore, l'errore quadratico sarebbe deciso
da una manciata di ticket dimenticati e di story esplose, e sceglieremmo gli iperparametri in base al
rumore invece che al comportamento tipico. È la scala in cui questi modelli lavorano, quindi è anche
quella su cui ha senso confrontarne le varianti. `make_scorer` trasforma la nostra funzione in uno
*scorer* per scikit-learn; il segno negativo serve perché scikit-learn massimizza sempre.

In [ ]:
risultati = []                                                     # una riga per modello, per il riepilogo finale
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)    # le stesse 5 "pieghe" per tutti i modelli


def rmse_log(y_vero, y_pred):
    '''RMSE in scala logaritmica: misura l'errore relativo tipico ed è poco sensibile agli outlier.
    Approssimativamente, 0.5 significa "sbagliamo in media di un fattore e^0.5 ≈ 1.65".'''
    return np.sqrt(mean_squared_error(np.log(y_vero), np.log(y_pred)))


scorer_log = make_scorer(rmse_log, greater_is_better=False)      # per GridSearchCV (segno negativo: sklearn massimizza)

## 3. Regressione lineare multivariata: `costo_cloud`

La regressione lineare stima il target come combinazione lineare delle feature:

$$\text{costo} = \beta_0 + \beta_1 \cdot \text{richieste} + \beta_2 \cdot \text{dati\_gb} + \beta_3 \cdot \text{deploy} + \beta_4 \cdot \text{indice\_mese} + \sum_s \gamma_s \cdot [\text{servizio} = s] + \varepsilon$$

I coefficienti $\beta$ si stimano minimizzando la somma dei quadrati dei residui (metodo dei minimi
quadrati, OLS). Qui il modello lineare non è solo un punto di partenza: una bolletta cloud *è* una
somma di voci a prezzo unitario, quindi i coefficienti hanno un significato diretto, in euro per
milione di richieste, per gigabyte, per deploy. Le dummy dei servizi catturano il costo fisso di
ciascuno; `indice_mese` il trend (aumenti di listino, crescita che non passa dalle altre variabili).

**Split temporale.** Questi dati sono una serie storica: uno split casuale userebbe mesi futuri per
prevedere mesi passati, cosa che in produzione non potremo mai fare. Addestriamo quindi sui primi
36 mesi e testiamo sugli ultimi 12. Non usiamo la cross-validazione a fold casuali per lo stesso
motivo.

**Due librerie, due scopi.** Con scikit-learn costruiamo il modello dentro una `Pipeline` e misuriamo
l'errore di previsione; con statsmodels stimiamo *lo stesso* modello per fare **inferenza**: quanto
sono precisi i coefficienti, quali sono significativi, i residui si comportano bene?

In [ ]:
feature_cloud = ["richieste_api_milioni", "dati_gb", "deploy", "indice_mese", "servizio"]

train_cloud = cloud[cloud["indice_mese"] <= 36].copy()      # primi 3 anni
test_cloud = cloud[cloud["indice_mese"] > 36].copy()        # ultimi 12 mesi
X_train_cloud, y_train_cloud = train_cloud[feature_cloud], train_cloud["costo_euro"]
X_test_cloud, y_test_cloud = test_cloud[feature_cloud], test_cloud["costo_euro"]
print(f"training: {len(train_cloud)} righe  ({train_cloud['mese'].min():%Y-%m} → {train_cloud['mese'].max():%Y-%m})")
print(f"test:     {len(test_cloud)} righe  ({test_cloud['mese'].min():%Y-%m} → {test_cloud['mese'].max():%Y-%m})")

# drop="first": una dummy in meno, il servizio di riferimento finisce nell'intercetta
# (stessa codifica "treatment" che userà statsmodels: così i coefficienti sono confrontabili)
prep_cloud = ColumnTransformer(
    [("cat", OneHotEncoder(drop="first"), ["servizio"])],
    remainder="passthrough", sparse_threshold=0, verbose_feature_names_out=False)

lineare_cloud = Pipeline([("prep", prep_cloud), ("ols", LinearRegression())])
lineare_cloud.fit(X_train_cloud, y_train_cloud)

pred_test_cloud = lineare_cloud.predict(X_test_cloud)
m = metriche(y_test_cloud, pred_test_cloud, "Regressione lineare")
m.update(dataset="costo_cloud", target="costo_euro (euro/mese)",
         RMSE_baseline=rmse_baseline(y_train_cloud, y_test_cloud))
risultati.append(m)
print(f"\nTest (ultimi 12 mesi): MSE = {m['MSE']:.0f}   RMSE = {m['RMSE']:.1f} euro   "
      f"(modello ingenuo: RMSE = {m['RMSE_baseline']:.1f} euro)")

print("\nCoefficienti stimati da scikit-learn (intercetta = {:.1f}):".format(lineare_cloud.named_steps["ols"].intercept_))
nomi_cloud = lineare_cloud.named_steps["prep"].get_feature_names_out()
pd.Series(lineare_cloud.named_steps["ols"].coef_, index=nomi_cloud).round(3)

scikit-learn ci dà i coefficienti, ma non ci dice **quanto fidarci**: nessun errore standard, nessun
intervallo di confidenza, nessun p-value. Per questo passiamo a statsmodels.

### 3.1 Inferenza con statsmodels

Usiamo l'interfaccia a formule (`smf.ols`), che ricorda R: `C(servizio)` crea automaticamente le
dummy, con il primo servizio in ordine alfabetico come riferimento. Partiamo di proposito da un
modello che include **anche** `utenti_attivi`, per vedere cosa succede quando due feature dicono
quasi la stessa cosa.

In [ ]:
formula_completa = ("costo_euro ~ richieste_api_milioni + utenti_attivi + dati_gb + deploy "
                    "+ indice_mese + C(servizio)")
ols_completo = smf.ols(formula_completa, data=train_cloud).fit()
print(ols_completo.summary())

**Come leggere il `summary`.** `coef` è la stima del coefficiente, `std err` la sua incertezza,
`t` il rapporto tra i due, `P>|t|` la probabilità di osservare un valore così estremo se il vero
coefficiente fosse zero, `[0.025 0.975]` l'intervallo di confidenza al 95%. `R-squared` è la quota di
varianza del costo spiegata dal modello; `Adj. R-squared` la corregge per il numero di variabili.

Notiamo che `utenti_attivi` e `richieste_api_milioni` hanno errori standard sproporzionati: il modello
non riesce a separare i loro effetti perché si muovono insieme. Lo misuriamo con il **VIF** (Variance
Inflation Factor): quante volte la varianza di un coefficiente è gonfiata dalla correlazione con le
altre feature. Valori sopra 5–10 segnalano un problema.

In [ ]:
def tabella_vif(risultato_ols):
    '''VIF di ogni regressore (esclusa l'intercetta) a partire dalla matrice del modello.'''
    X = risultato_ols.model.exog
    nomi = risultato_ols.model.exog_names
    vif = pd.DataFrame({"variabile": nomi,
                        "VIF": [variance_inflation_factor(X, i) for i in range(X.shape[1])]})
    return vif[vif["variabile"] != "Intercept"].round(2)

tabella_vif(ols_completo)

Teniamo `richieste_api_milioni` (è la grandezza che il provider fattura davvero) e togliamo
`utenti_attivi`. Questo è il modello finale, lo stesso della pipeline scikit-learn.

In [ ]:
formula_cloud = "costo_euro ~ richieste_api_milioni + dati_gb + deploy + indice_mese + C(servizio)"
ols_cloud = smf.ols(formula_cloud, data=train_cloud).fit()
print(ols_cloud.summary())
print("\nVIF dopo aver rimosso utenti_attivi:")
print(tabella_vif(ols_cloud).to_string(index=False))

I dati sono sintetici, quindi conosciamo il "listino" con cui sono stati generati: 9 euro per milione
di richieste, 0,045 euro per GB, 25 euro per deploy. Confrontiamo le stime con i valori veri: è il
modo più diretto per capire cosa sono un errore standard e un intervallo di confidenza.

In [ ]:
listino = {"richieste_api_milioni": 9.0, "dati_gb": 0.045, "deploy": 25.0}
voci = list(listino)
confronto = pd.DataFrame({
    "stima": ols_cloud.params[voci],
    "err_std": ols_cloud.bse[voci],
    "IC95_inf": ols_cloud.conf_int().loc[voci, 0],
    "IC95_sup": ols_cloud.conf_int().loc[voci, 1],
    "p_value": ols_cloud.pvalues[voci],
    "valore_vero": pd.Series(listino),
})
confronto.round(4)

In [ ]:
# Lo stesso modello, due librerie: le previsioni sul test coincidono
pred_sm = ols_cloud.predict(test_cloud)
print(f"RMSE test con statsmodels:   {np.sqrt(mean_squared_error(y_test_cloud, pred_sm)):.2f} euro")
print(f"RMSE test con scikit-learn:  {np.sqrt(mean_squared_error(y_test_cloud, pred_test_cloud)):.2f} euro")

### 3.2 Diagnostica dei residui

L'inferenza OLS poggia su alcune ipotesi: residui a media zero, con varianza costante, senza
struttura, approssimativamente normali. Le verifichiamo con tre grafici:

- **residui vs valori stimati**: non devono mostrare forme (curve, imbuti);
- **Q-Q plot** dei residui studentizzati: se i punti seguono la diagonale, i residui sono normali;
- **residui nel tempo**: fondamentale per una serie storica; una deriva sistematica segnala che
  manca qualcosa (un aumento di listino, un cambio di architettura).

I residui studentizzati sono i residui divisi per la loro deviazione standard stimata: valori oltre
±3 sono candidati outlier.

In [ ]:
influenza = ols_cloud.get_influence()
resid_stud = pd.Series(influenza.resid_studentized_internal, index=train_cloud.index)

fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))
axes[0].scatter(ols_cloud.fittedvalues, ols_cloud.resid, alpha=0.6, s=15)
axes[0].axhline(0, color="red")
axes[0].set_title("Residui vs valori stimati")
axes[0].set_xlabel("costo stimato (euro)")
axes[0].set_ylabel("residuo (euro)")
sm.qqplot(resid_stud, line="45", ax=axes[1])
axes[1].set_title("Q-Q plot dei residui studentizzati")
axes[2].scatter(train_cloud["mese"], ols_cloud.resid, alpha=0.6, s=15)
axes[2].axhline(0, color="red")
axes[2].set_title("Residui nel tempo (training)")
axes[2].tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

anomali = resid_stud.abs() > 3
print(f"Osservazioni con |residuo studentizzato| > 3: {anomali.sum()}")
train_cloud.loc[anomali, ["mese", "servizio", "costo_euro"]].assign(
    costo_stimato=ols_cloud.fittedvalues[anomali].round(0),
    residuo_stud=resid_stud[anomali].round(2))

I punti fuori scala sono i **mesi anomali** intravisti nell'analisi esplorativa: costi da 1,5 a 2,5
volte il previsto, senza che nessuna feature lo giustifichi (un job impazzito, un test di carico
dimenticato). Sono proprio i casi che, in un'azienda, vale la pena andare a investigare: la
regressione qui funziona come **rilevatore di anomalie**.

Sono pochi ma pesano molto sui minimi quadrati. Proviamo a stimare di nuovo il modello senza di
loro e osserviamo l'effetto sulle stime dei prezzi unitari. Il test set invece lo lasciamo com'è:
in produzione i mesi anomali arriveranno comunque, e l'errore sul test deve tenerne conto.

In [ ]:
train_pulito = train_cloud[~anomali]
ols_pulito = smf.ols(formula_cloud, data=train_pulito).fit()

confronto_pulito = pd.DataFrame({
    "stima_con_anomalie": ols_cloud.params[voci],
    "stima_senza_anomalie": ols_pulito.params[voci],
    "err_std_con": ols_cloud.bse[voci],
    "err_std_senza": ols_pulito.bse[voci],
    "valore_vero": pd.Series(listino),
})
print(f"R² aggiustato: con anomalie {ols_cloud.rsquared_adj:.3f}  →  senza anomalie {ols_pulito.rsquared_adj:.3f}")
pred_pulito = ols_pulito.predict(test_cloud)
print(f"RMSE test: modello con anomalie {np.sqrt(mean_squared_error(y_test_cloud, pred_sm)):.1f} euro"
      f"  →  senza anomalie {np.sqrt(mean_squared_error(y_test_cloud, pred_pulito)):.1f} euro")
confronto_pulito.round(4)

In [ ]:
# Residui sul test: errori sistematici sugli ultimi 12 mesi?
fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(test_cloud["mese"], y_test_cloud - pred_pulito, alpha=0.7)
ax.axhline(0, color="red")
ax.set_title("Residui sul test (ultimi 12 mesi): reale − previsto")
ax.set_ylabel("euro")
plt.show()

**Cosa portiamo a casa.**

- I coefficienti stimati ricostruiscono il listino del provider con buona precisione, e gli
  intervalli di confidenza contengono i valori veri. Senza gli outlier gli errori standard si
  riducono sensibilmente: pochi punti estremi possono spostare le stime più di centinaia di punti
  "normali".
- Le dummy dei servizi assorbono i costi fissi; `indice_mese` assorbe il trend, compresi gli
  aumenti di listino intervenuti nel periodo di training.
- I residui sul test tendono ad avere segno positivo negli ultimi mesi: il modello, addestrato fino
  ad agosto 2025, non conosce l'ultimo rincaro di listino e sottostima leggermente. È il fenomeno
  della **deriva** (*drift*): un modello va ri-addestrato periodicamente, e i residui nel tempo sono
  il modo per accorgersene.

## 4. Albero decisionale: `ticket_risoluzione`

Un albero di regressione divide ripetutamente i dati con domande del tipo "la priorità è almeno
Alta?", "la descrizione supera 800 caratteri?", scegliendo ogni volta la domanda che riduce di più
l'errore quadratico. Ogni foglia predice la media del target dei ticket che vi finiscono. Il
risultato è leggibile come una catena di `if/else`: per un pubblico di sviluppatori è il modello
più trasparente che esista.

Perché questo dataset:

- le variabili più informative sono **categoriche** e l'albero le usa direttamente, senza scaling;
- gli effetti sono **combinazioni** (una richiesta di funzionalità a bassa priorità su Infrastruttura),
  che l'albero cattura per costruzione;
- la priorità è ordinale: la codifichiamo con `OrdinalEncoder` rispettando l'ordine, così una sola
  soglia ("priorità ≥ Alta") basta a separare i ticket urgenti.

Il rischio tipico dell'albero è il **sovra-adattamento**: lasciato crescere senza limiti, memorizza
il training e generalizza male. Lo controlliamo con due iperparametri, la profondità massima
(`max_depth`) e il numero minimo di osservazioni per foglia (`min_samples_leaf`), scelti in
cross-validazione. Addestriamo su `log(ore)` perché, con un target così asimmetrico, l'errore
quadratico in ore sarebbe dominato dai pochi ticket dimenticati e l'albero costruirebbe rami
appositamente per loro; per la stessa ragione la ricerca degli iperparametri usa `scorer_log`.

In [ ]:
feature_ticket = ["priorita", "componente", "tipo", "lunghezza_descrizione", "commenti_24h",
                  "carico_assegnatario", "esperienza_assegnatario_mesi", "allegati"]
X_ticket, y_ticket = ticket[feature_ticket], ticket["ore_risoluzione"]
X_train_ticket, X_test_ticket, y_train_ticket, y_test_ticket = train_test_split(
    X_ticket, y_ticket, test_size=0.2, random_state=RANDOM_STATE)
print(f"training: {len(X_train_ticket)} ticket   test: {len(X_test_ticket)} ticket")

prep_ticket = ColumnTransformer([
    ("ordinale", OrdinalEncoder(categories=[["Bassa", "Media", "Alta", "Critica"]]), ["priorita"]),
    ("onehot", OneHotEncoder(handle_unknown="ignore"), ["componente", "tipo"]),
    ("num", "passthrough", ["lunghezza_descrizione", "commenti_24h", "carico_assegnatario",
                            "esperienza_assegnatario_mesi", "allegati"]),
], sparse_threshold=0, verbose_feature_names_out=False)

albero_ticket = TransformedTargetRegressor(
    regressor=Pipeline([("prep", prep_ticket),
                        ("albero", DecisionTreeRegressor(random_state=RANDOM_STATE))]),
    func=np.log, inverse_func=np.exp)

griglia_albero = {
    "regressor__albero__max_depth": [2, 3, 4, 5, 6, 8, 10, 15, None],
    "regressor__albero__min_samples_leaf": [1, 5, 10, 20, 40],
}
ricerca_albero = GridSearchCV(albero_ticket, griglia_albero, cv=cv, scoring=scorer_log,
                              return_train_score=True, n_jobs=-1)
ricerca_albero.fit(X_train_ticket, y_train_ticket)
print("Iperparametri migliori:", ricerca_albero.best_params_)
print(f"RMSE in cross-validazione (5 fold, scala log): {-ricerca_albero.best_score_:.3f}")

Prima di guardare il test, visualizziamo il compromesso tra sotto- e sovra-adattamento: RMSE sul
training e in cross-validazione al variare della profondità, a parità di `min_samples_leaf`
(quello scelto dalla ricerca). L'errore di training scende sempre; quello di validazione no.

In [ ]:
ris_albero = pd.DataFrame(ricerca_albero.cv_results_)
foglia_migliore = ricerca_albero.best_params_["regressor__albero__min_samples_leaf"]
curva = ris_albero[ris_albero["param_regressor__albero__min_samples_leaf"] == foglia_migliore].copy()
curva["profondita"] = curva["param_regressor__albero__max_depth"].astype(float).fillna(25).astype(int)
curva = curva.sort_values("profondita")

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(curva["profondita"], -curva["mean_train_score"], "o-", label="RMSE training")
ax.plot(curva["profondita"], -curva["mean_test_score"], "o-", label="RMSE cross-validazione")
ax.set_xlabel("profondità massima (25 = nessun limite)")
ax.set_ylabel("RMSE (scala log)")
ax.set_title(f"Albero con min_samples_leaf = {foglia_migliore}: sotto-adattamento a sinistra, sovra-adattamento a destra")
ax.legend()
plt.show()

In [ ]:
pred_test_ticket = ricerca_albero.predict(X_test_ticket)      # usa automaticamente il modello migliore
m = metriche(y_test_ticket, pred_test_ticket, "Albero decisionale")
m.update(dataset="ticket_risoluzione", target="ore_risoluzione (ore)",
         RMSE_baseline=rmse_baseline(y_train_ticket, y_test_ticket))
risultati.append(m)
print(f"Test: MSE = {m['MSE']:.0f}   RMSE = {m['RMSE']:.1f} ore   (modello ingenuo: RMSE = {m['RMSE_baseline']:.1f} ore)")

# Due lenti complementari, con la stessa pietra di paragone: il modello ingenuo che predice
# la media geometrica (scala log) o la mediana (errore assoluto) del training
ingenuo_log = np.full(len(y_test_ticket), np.exp(np.log(y_train_ticket).mean()))
ingenuo_med = np.full(len(y_test_ticket), y_train_ticket.median())
print(f"RMSE in scala log:        {rmse_log(y_test_ticket, pred_test_ticket):.3f}   (modello ingenuo: {rmse_log(y_test_ticket, ingenuo_log):.3f})")
print(f"Errore assoluto mediano:  {np.median(np.abs(y_test_ticket - pred_test_ticket)):.1f} ore   "
      f"(modello ingenuo: {np.median(np.abs(y_test_ticket - ingenuo_med)):.1f} ore)")
print(f"Ticket più lungo nel test: {y_test_ticket.max():.0f} ore, previsto a {pred_test_ticket[np.argmax(y_test_ticket.values)]:.0f} ore")

L'RMSE in ore è **dominato dai ticket dimenticati**: un solo ticket da quasi 2.000 ore, previsto a
poche decine, contribuisce all'MSE più di tutti gli altri messi insieme, e nessuna feature del
dataset permette di prevederlo. Per questo l'RMSE in ore migliora poco rispetto al modello ingenuo,
mentre le altre due lenti, RMSE in scala log ed errore assoluto mediano, mostrano che sul ticket
"tipico" l'albero sbaglia molto meno della media. È una lezione generale: MSE e RMSE vanno letti
insieme alla distribuzione del target, e quando ci sono outlier che il modello non può spiegare
conviene affiancare una metrica robusta.

### 4.1 Leggere l'albero

Stampiamo le prime tre livelli di domande come testo (i valori nelle foglie sono in `log(ore)`:
`exp(3.7) ≈ 40` ore) e poi disegniamo l'albero. Infine guardiamo l'**importanza delle feature**: la
quota di riduzione dell'errore attribuibile a ciascuna variabile in tutte le divisioni dell'albero.

In [ ]:
pipe_albero = ricerca_albero.best_estimator_.regressor_       # la pipeline addestrata su log(ore)
albero = pipe_albero.named_steps["albero"]
nomi_ticket = list(pipe_albero.named_steps["prep"].get_feature_names_out())
print(f"Profondità effettiva dell'albero: {albero.get_depth()}, foglie: {albero.get_n_leaves()}\n")
print(export_text(albero, feature_names=nomi_ticket, max_depth=3, decimals=2))

In [ ]:
fig, ax = plt.subplots(figsize=(24, 9))
plot_tree(albero, feature_names=nomi_ticket, max_depth=3, filled=True, rounded=True,
          impurity=False, fontsize=9, ax=ax)
ax.set_title("Albero decisionale (primi 3 livelli; value = log(ore) medio della foglia)")
plt.show()

importanze = pd.Series(albero.feature_importances_, index=nomi_ticket).sort_values()
fig, ax = plt.subplots(figsize=(8, 5))
importanze.plot.barh(ax=ax, color="steelblue")
ax.set_title("Importanza delle feature")
ax.set_xlabel("quota di riduzione dell'errore")
plt.show()

**Cosa portiamo a casa.**

- Le prime divisioni riguardano priorità e tipo: l'albero ha "scoperto" da solo la gerarchia che
  avevamo visto nei boxplot, e la esprime come regole verificabili con chi gestisce i ticket.
- La priorità codificata come ordinale viene divisa con una sola soglia; se l'avessimo codificata
  one-hot, l'albero avrebbe avuto bisogno di più domande per lo stesso risultato.
- La curva di profondità mostra il compromesso tipico: sotto una certa profondità il modello è
  troppo semplice, sopra memorizza il rumore. La cross-validazione sceglie il punto di equilibrio,
  senza toccare il test.
- L'albero fa previsioni "a gradini" (una costante per foglia): non estrapola e non cattura trend
  lisci. Quando servono, gli insiemi di alberi (random forest, gradient boosting) sono il passo
  successivo naturale.

## 5. k-Nearest Neighbors: `story_point_ore`

Il kNN non "impara" una formula: per stimare una nuova story cerca nel training le **k story più
simili** (quelle a distanza minima nello spazio delle feature) e ne restituisce la media delle ore,
eventualmente pesata per la vicinanza (`weights="distance"`). È il ragionamento che ogni team fa a
voce durante una stima: "l'ultima volta che abbiamo fatto una story da 5 punti con una dipendenza
esterna ci abbiamo messo tre giorni".

Perché questo dataset: le feature sono poche, il concetto di "story simile" è naturale e la
relazione punti → ore è non lineare (il kNN non ha bisogno di specificarla). Due accorgimenti sono
però indispensabili:

- **scaling**: la distanza mescola unità diverse (punti, numero di criteri, dummy 0/1); senza
  standardizzare, la variabile con i numeri più grandi dominerebbe la distanza;
- **niente feature irrilevanti**: `sprint` allontanerebbe story identiche solo perché fatte in
  momenti diversi. L'abbiamo escluso in fase esplorativa.

L'iperparametro chiave è **k**: con k piccolo il modello segue il rumore (alta varianza), con k
grande appiattisce tutto verso la media (alto bias). Lo scegliamo in cross-validazione, insieme al
tipo di pesatura. Anche qui lavoriamo su `log(ore)`: la media dei vicini in scala log è una media
geometrica, meno sensibile alle story esplose.

In [ ]:
feature_story = ["story_point", "criteri_accettazione", "dipendenze_esterne",
                 "team", "tipo", "seniority_sviluppatore"]
X_story, y_story = story[feature_story], story["ore_effettive"]
X_train_story, X_test_story, y_train_story, y_test_story = train_test_split(
    X_story, y_story, test_size=0.2, random_state=RANDOM_STATE)
print(f"training: {len(X_train_story)} story   test: {len(X_test_story)} story")

prep_story = ColumnTransformer([
    ("num", StandardScaler(), ["story_point", "criteri_accettazione", "dipendenze_esterne"]),
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["team", "tipo", "seniority_sviluppatore"]),
], sparse_threshold=0, verbose_feature_names_out=False)

knn_story = TransformedTargetRegressor(
    regressor=Pipeline([("prep", prep_story), ("knn", KNeighborsRegressor())]),
    func=np.log, inverse_func=np.exp)

griglia_knn = {"regressor__knn__n_neighbors": list(range(1, 41)),
               "regressor__knn__weights": ["uniform", "distance"]}
ricerca_knn = GridSearchCV(knn_story, griglia_knn, cv=cv, scoring=scorer_log, n_jobs=-1)
ricerca_knn.fit(X_train_story, y_train_story)
print("Iperparametri migliori:", ricerca_knn.best_params_)
print(f"RMSE in cross-validazione (5 fold, scala log): {-ricerca_knn.best_score_:.3f}")

ris_knn = pd.DataFrame(ricerca_knn.cv_results_)
fig, ax = plt.subplots(figsize=(9, 4.5))
for pesi in ["uniform", "distance"]:
    sotto = ris_knn[ris_knn["param_regressor__knn__weights"] == pesi]
    ax.plot(sotto["param_regressor__knn__n_neighbors"].astype(int), -sotto["mean_test_score"],
            "o-", markersize=3, label=f"weights = {pesi}")
ax.set_xlabel("k (numero di vicini)")
ax.set_ylabel("RMSE cross-validazione (scala log)")
ax.set_title("Scelta di k: varianza alta a sinistra, bias alto a destra")
ax.legend()
plt.show()

In [ ]:
pred_test_story = ricerca_knn.predict(X_test_story)
m = metriche(y_test_story, pred_test_story, "k-Nearest Neighbors")
m.update(dataset="story_point_ore", target="ore_effettive (ore)",
         RMSE_baseline=rmse_baseline(y_train_story, y_test_story))
risultati.append(m)
print(f"Test: MSE = {m['MSE']:.0f}   RMSE = {m['RMSE']:.1f} ore   (modello ingenuo: RMSE = {m['RMSE_baseline']:.1f} ore)")
ingenuo_log = np.full(len(y_test_story), np.exp(np.log(y_train_story).mean()))
print(f"RMSE in scala log: {rmse_log(y_test_story, pred_test_story):.3f}   (modello ingenuo: {rmse_log(y_test_story, ingenuo_log):.3f})")

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(y_test_story, pred_test_story, alpha=0.5, s=15)
lim = [0, y_test_story.max() * 1.05]
ax.plot(lim, lim, "r--")
ax.set_xlabel("ore effettive reali")
ax.set_ylabel("ore previste dal kNN")
ax.set_title("kNN: previsto vs reale (test)")
plt.show()

### 5.1 Guardare i vicini

Il vantaggio del kNN è che ogni previsione si può **spiegare mostrando i vicini**. Prendiamo una
story del test, chiediamo al modello quali sono le 5 story del training più vicine e confrontiamo le
loro ore con la previsione. Per farlo estraiamo dalla pipeline addestrata il preprocessore (per
trasformare la nuova story nello stesso spazio) e il regressore kNN (metodo `kneighbors`).

In [ ]:
pipe_knn = ricerca_knn.best_estimator_.regressor_
prep_addestrato, knn_addestrato = pipe_knn.named_steps["prep"], pipe_knn.named_steps["knn"]

esempio = X_test_story.iloc[[0]]
distanze, indici = knn_addestrato.kneighbors(prep_addestrato.transform(esempio), n_neighbors=5)

print("Story da stimare (dal test set):")
print(esempio.to_string(index=False))
print(f"\nOre effettive reali: {y_test_story.iloc[0]:.1f}   |   previsione del kNN: {ricerca_knn.predict(esempio)[0]:.1f} ore")
print("\nLe 5 story del training più simili (distanza 0 = feature identiche):")
X_train_story.iloc[indici[0]].assign(ore_effettive=y_train_story.iloc[indici[0]].values,
                                     distanza=distanze[0].round(3))

**Cosa portiamo a casa.**

- Il kNN batte nettamente il modello ingenuo senza che gli abbiamo detto nulla sulla forma della
  relazione punti → ore: la non linearità è "contenuta" nei vicini.
- Molte story hanno feature identiche (stessi punti, stesso team, stessa seniority...): con k = 1 la
  previsione dipende da *quale* delle copie viene scelta, ed è instabile; con il k scelto dalla
  ricerca (una decina di vicini, pesati per distanza) facciamo la media di un piccolo gruppo di story
  simili, che è esattamente ciò che vorremmo da una stima. Oltre, i vicini diventano story diverse e
  la previsione si appiattisce: la curva risale.
- Il prezzo del kNN è che non produce un modello compatto: per ogni previsione deve tenere in memoria
  e scandire tutto il training. Su 800 righe è istantaneo, su milioni no.
- Il grafico previsto/reale mostra la dispersione crescente con le ore: la variabilità delle story
  grandi resta grande anche per il modello migliore. Non è un difetto dell'algoritmo, è
  informazione: la stima di una story da 13 punti dovrebbe sempre essere accompagnata da un
  intervallo, non da un numero secco.

## 6. Rete neurale MLP: `build_ci`

Un percettrone multistrato (MLP) è una sequenza di strati di neuroni: ogni neurone calcola una
combinazione lineare dei suoi ingressi e la passa a una funzione di attivazione non lineare (qui la
ReLU). Impilando gli strati la rete può approssimare relazioni arbitrarie, interazioni comprese.
I pesi vengono appresi minimizzando l'errore quadratico con la discesa del gradiente (qui
l'ottimizzatore Adam).

Perché questo dataset: la durata di una build dipende da **interazioni** (la cache miss pesa di più
sui commit grandi) e da **effetti non monotoni** (le ore di punta), con 2.000 osservazioni e feature
quasi tutte numeriche. Un modello lineare qui è limitato per costruzione; la rete no.

Scelte di preparazione:

- `log1p` + standardizzazione per `righe_modificate` e `file_modificati` (code lunghe), standardizzazione
  semplice per `moduli_toccati` e `numero_test`; `cache_hit` è già 0/1;
- one-hot per `ora_giorno` (l'effetto dell'ora non è monotono: trattarla come numero costringerebbe
  la rete a imparare una forma complicata), `giorno_settimana`, `runner`, `trigger`;
- target lasciato in **minuti**: non ci sono outlier estremi, e vogliamo vedere se la rete impara
  da sola la struttura moltiplicativa dei dati, senza che gliela suggeriamo con una trasformazione.

Scelte di addestramento: due strati nascosti (64 e 32 neuroni), **early stopping** (la rete
mette da parte il 15% del training come validazione interna e si ferma quando l'errore di
validazione smette di migliorare: è il principale antidoto al sovra-adattamento), penalità L2
leggera (`alpha`). Prima di fidarci del risultato confrontiamo, in cross-validazione, la rete con
una regressione lineare addestrata sugli stessi identici dati: solo così sappiamo se la complessità
aggiunta sta pagando.

In [ ]:
feature_build = ["righe_modificate", "file_modificati", "moduli_toccati", "numero_test", "cache_hit",
                 "ora_giorno", "giorno_settimana", "runner", "trigger"]
X_build, y_build = build[feature_build], build["durata_minuti"]
X_train_build, X_test_build, y_train_build, y_test_build = train_test_split(
    X_build, y_build, test_size=0.2, random_state=RANDOM_STATE)
print(f"training: {len(X_train_build)} build   test: {len(X_test_build)} build")

prep_build = ColumnTransformer([
    ("num_log", Pipeline([("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
                          ("scala", StandardScaler())]), ["righe_modificate", "file_modificati"]),
    ("num", StandardScaler(), ["moduli_toccati", "numero_test"]),
    ("bin", "passthrough", ["cache_hit"]),
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["ora_giorno", "giorno_settimana", "runner", "trigger"]),
], sparse_threshold=0, verbose_feature_names_out=False)


def modello_build(regressore):
    '''Stessa preparazione per qualunque regressore: confronto equo.'''
    return Pipeline([("prep", prep_build), ("reg", regressore)])


lineare_build = modello_build(LinearRegression())
mlp_build = modello_build(MLPRegressor(hidden_layer_sizes=(64, 32), activation="relu", solver="adam",
                                       alpha=1e-3, learning_rate_init=1e-3, max_iter=3000,
                                       early_stopping=True, validation_fraction=0.15,
                                       n_iter_no_change=50, random_state=RANDOM_STATE))

for nome, modello in [("Regressione lineare (confronto)", lineare_build), ("Rete neurale MLP", mlp_build)]:
    punteggi = cross_val_score(modello, X_train_build, y_train_build, cv=cv,
                               scoring="neg_root_mean_squared_error", n_jobs=-1)
    print(f"{nome:32s} RMSE in cross-validazione = {-punteggi.mean():.2f} ± {punteggi.std():.2f} minuti"
          f"   (per piega: {np.round(-punteggi, 2)})")

La rete riduce l'errore rispetto al modello lineare a parità di dati e preparazione, e lo fa in
modo consistente sulle cinque pieghe: la differenza è il valore delle non linearità che ha imparato.
Addestriamo ora la rete definitiva su tutto il training e osserviamo la **curva di apprendimento**:
la loss di training (MSE in minuti²) e il punteggio sulla validazione interna, epoca per epoca, fino
allo stop anticipato.

In [ ]:
mlp_build.fit(X_train_build, y_train_build)
rete = mlp_build.named_steps["reg"]
print(f"Epoche eseguite: {rete.n_iter_} su un massimo di {rete.max_iter} (early stopping)")
print(f"Migliore R² sulla validazione interna: {rete.best_validation_score_:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
axes[0].plot(rete.loss_curve_)
axes[0].set_yscale("log")                      # scala log: altrimenti si vede solo il crollo iniziale
axes[0].set_title("Loss di training per epoca (MSE, minuti²; scala log)")
axes[0].set_xlabel("epoca")
axes[0].set_ylabel("loss")
axes[1].plot(rete.validation_scores_, color="darkorange")
axes[1].set_ylim(0, 1)
axes[1].set_title("R² sulla validazione interna per epoca")
axes[1].set_xlabel("epoca")
axes[1].set_ylabel("R²")
plt.tight_layout()
plt.show()

In [ ]:
lineare_build.fit(X_train_build, y_train_build)
pred_test_lineare = lineare_build.predict(X_test_build)
pred_test_mlp = mlp_build.predict(X_test_build)

m_lin = metriche(y_test_build, pred_test_lineare, "Regressione lineare (confronto)")
m = metriche(y_test_build, pred_test_mlp, "Rete neurale MLP")
m.update(dataset="build_ci", target="durata_minuti (minuti)",
         RMSE_baseline=rmse_baseline(y_train_build, y_test_build))
risultati.append(m)
print(f"Regressione lineare (confronto): MSE = {m_lin['MSE']:.2f}   RMSE = {m_lin['RMSE']:.2f} minuti")
print(f"Rete neurale MLP:                MSE = {m['MSE']:.2f}   RMSE = {m['RMSE']:.2f} minuti")
print(f"Modello ingenuo (media):         RMSE = {m['RMSE_baseline']:.2f} minuti")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(y_test_build, pred_test_mlp, alpha=0.5, s=15)
lim = [0, y_test_build.max() * 1.05]
axes[0].plot(lim, lim, "r--")
axes[0].set_xlabel("durata reale (minuti)")
axes[0].set_ylabel("durata prevista (minuti)")
axes[0].set_title("MLP: previsto vs reale (test)")
per_ora = pd.DataFrame({"reale": y_test_build.values, "prevista dalla rete": pred_test_mlp,
                        "ora": X_test_build["ora_giorno"].values}).groupby("ora").mean()
per_ora.plot(ax=axes[1], marker="o")
axes[1].set_title("Durata media per ora di avvio (test): la rete ha imparato le ore di punta")
axes[1].set_ylabel("minuti")
plt.tight_layout()
plt.show()

**Cosa portiamo a casa.**

- A parità di dati e preparazione, la rete riduce l'errore rispetto al modello lineare perché
  rappresenta le interazioni e la struttura moltiplicativa presenti nei dati. La cross-validazione ci
  ha permesso di affermarlo prima di guardare il test.
- Il vantaggio non è "magico": se addestrassimo il modello lineare su `log(durata)`, cioè se fossimo
  noi a indovinare la trasformazione giusta, recupererebbe gran parte del divario (vale la pena
  provarlo come esercizio). La rete ci evita di dover conoscere in anticipo la forma della relazione;
  la conoscenza del dominio, quando c'è, resta una scorciatoia potentissima.
- L'early stopping ha fermato l'addestramento ben prima del massimo di epoche: la curva di
  validazione smette di salire mentre la loss di training continua a scendere, il segno del
  sovra-adattamento che stiamo evitando.
- Il prezzo è l'opacità: la rete non ci dice *perché* una build è lenta. Se per il team conta più
  capire che prevedere, un albero o un modello lineare con interazioni esplicite possono essere una
  scelta migliore anche a costo di qualche minuto di RMSE in più.
- Nota pratica: le reti sono sensibili all'inizializzazione (`random_state`) e allo scaling; senza
  standardizzazione la stessa architettura converge male o non converge affatto. Vale la pena
  provarlo come esercizio, togliendo lo `StandardScaler` dalla pipeline.

## 7. Riepilogo

Raccogliamo le metriche di test dei quattro modelli. Le unità sono quelle di ciascun target, quindi
gli RMSE **non sono confrontabili tra dataset**: il confronto sensato è, per ogni riga, con il
modello ingenuo (colonna `RMSE_baseline`). La riga dei ticket va letta con quanto visto nella
sezione 4: l'RMSE in ore è deciso da pochi ticket dimenticati che nessun modello può prevedere,
mentre sul ticket tipico l'albero è nettamente migliore del modello ingenuo.

In [ ]:
tabella = pd.DataFrame(risultati)[["modello", "dataset", "target", "MSE", "RMSE", "RMSE_baseline"]]
tabella["riduzione_RMSE_vs_ingenuo"] = (1 - tabella["RMSE"] / tabella["RMSE_baseline"]).map("{:.0%}".format)
tabella.round(2)

**Le lezioni trasversali.**

1. **Guardare i dati prima del modello.** Tutte le scelte importanti (target in scala log, codifica
   ordinale della priorità, esclusione di `sprint`, split temporale per i costi cloud) sono nate
   dall'analisi esplorativa, non dall'algoritmo.
2. **Una pipeline per ogni modello.** Trasformazioni stimate sul solo training e riapplicate al
   test: è l'unico modo per avere metriche oneste. `TransformedTargetRegressor` estende la stessa
   disciplina al target.
3. **Cross-validazione per scegliere, test set per giudicare.** Gli iperparametri (profondità
   dell'albero, k del kNN, architettura della rete) si scelgono sul training; il test si guarda una
   volta sola, alla fine.
4. **MSE e RMSE non bastano da soli.** Vanno letti contro una baseline e insieme alla distribuzione
   del target: con code lunghe l'RMSE racconta gli outlier, non il caso tipico.
5. **Ogni algoritmo ha il suo terreno.** Il modello lineare quando la relazione è davvero lineare e
   servono coefficienti interpretabili (e statsmodels per l'inferenza); l'albero quando contano regole
   e variabili categoriche; il kNN quando "casi simili" è la domanda giusta; la rete neurale quando
   ci sono interazioni e non linearità e abbiamo dati a sufficienza.

**Esercizi possibili.** Applicare ogni algoritmo a un dataset diverso da quello scelto qui e
spiegare perché funziona meglio o peggio; sostituire l'albero con una random forest; aggiungere
alla regressione lineare del cloud un termine per gli aumenti di listino; provare il kNN con e senza
`sprint` tra le feature; togliere lo scaling alla rete neurale e osservare cosa succede.